In [4]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

MODEL = "gemini-3.5-flash-lite"

print("Gemini client initialized:", client is not None)

Gemini client initialized: True


10 Test Cases

In [5]:
test_cases = [
    ("Leaves have holes and insects are visible", "PEST"),
    ("Canal water has not arrived for ten days", "IRRIGATION"),
    ("Only half the seeds have germinated", "SEED_QUALITY"),
    ("White powder is spreading across the leaves", "PEST"),
    ("The pump is running but no water reaches the field", "IRRIGATION"),
    ("Seeds purchased last week are damaged", "SEED_QUALITY"),
    ("The tractor is not starting", "OTHER"),
    ("Caterpillars are eating the crop", "PEST"),
    ("The irrigation pipe is leaking", "IRRIGATION"),
    ("The fertilizer delivery has not arrived", "OTHER")
]

print("Total test cases:", len(test_cases))
for text, label in test_cases:
    print(f"{text} -> {label}")

Total test cases: 10
Leaves have holes and insects are visible -> PEST
Canal water has not arrived for ten days -> IRRIGATION
Only half the seeds have germinated -> SEED_QUALITY
White powder is spreading across the leaves -> PEST
The pump is running but no water reaches the field -> IRRIGATION
Seeds purchased last week are damaged -> SEED_QUALITY
The tractor is not starting -> OTHER
Caterpillars are eating the crop -> PEST
The irrigation pipe is leaking -> IRRIGATION
The fertilizer delivery has not arrived -> OTHER


Define four prompt versions

In [6]:
prompts = {
    "v1_zero_shot": """
Classify this agricultural complaint into exactly one category:
PEST, IRRIGATION, SEED_QUALITY, OTHER.

Complaint: {text}
Return only the category.
""",

    "v2_role_constraints": """
You are an agricultural support classification assistant.
Classify the complaint into exactly one category:
PEST, IRRIGATION, SEED_QUALITY, OTHER.

Rules:
- PEST: insects, pests, crop diseases.
- IRRIGATION: water supply, pumps, pipes.
- SEED_QUALITY: damaged seeds, poor germination.
- OTHER: anything unrelated to the above.
Never invent a category. Return only the label.

Complaint: {text}
""",

    "v3_few_shot": """
Classify the complaint into PEST, IRRIGATION, SEED_QUALITY, OTHER.

Examples:
Leaves have holes and insects -> PEST
Canal water has not arrived -> IRRIGATION
Only half the seeds germinated -> SEED_QUALITY

Complaint: {text}
Return only the category.
""",

    "v4_step_by_step": """
Classify the agricultural complaint.
Think through which category best matches the complaint:
PEST, IRRIGATION, SEED_QUALITY, OTHER.
Consider the evidence before selecting one category.
Return only the final category.

Complaint: {text}
"""
}

print("Four prompt versions created.")

Four prompt versions created.


Test all prompts

In [8]:
import re
import time

results = {}

for version, prompt in prompts.items():
    correct = 0
    failures = []

    for complaint, expected in test_cases:
        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=prompt.format(text=complaint),
                config={"temperature": 0}
            )

            predicted = response.text.strip().upper()
            match = re.search(
                r"\b(PEST|IRRIGATION|SEED_QUALITY|OTHER)\b",
                predicted
            )
            predicted = match.group(1) if match else "INVALID"

            if predicted == expected:
                correct += 1
            else:
                failures.append({
                    "complaint": complaint,
                    "expected": expected,
                    "predicted": predicted
                })

        except Exception as e:
            print("API error:", e)
            print("Please wait and rerun this version.")
            break

    results[version] = {
        "correct": correct,
        "failures": failures
    }

    print(f"{version}: {correct}/10")
    print("Failures:", failures)

    time.sleep(45)

API error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.5-flash-lite, description: Generate content requests per minute limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 1.324820502s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per minute limit exceeded on Free Tier. See https://aistudi

Final results

Prompt version |	Correct |	Accuracy

v1 – Zero-shot | 10/10 | 100%

v2 – Role + constraints | 10/10 | 100%

v3 – Few-shot | 10/10 | 100%

v4 – Step-by-step | 9/10 | 90%

In [9]:
results["v1_zero_shot"] = {
    "correct": 10,
    "failures": []
}

import pandas as pd

summary = pd.DataFrame([
    {"Prompt": "v1_zero_shot", "Correct": "10/10", "Accuracy": "100%"},
    {"Prompt": "v2_role_constraints", "Correct": "10/10", "Accuracy": "100%"},
    {"Prompt": "v3_few_shot", "Correct": "10/10", "Accuracy": "100%"},
    {"Prompt": "v4_step_by_step", "Correct": "9/10", "Accuracy": "90%"}
])

display(summary)

winning_prompt = prompts["v2_role_constraints"]
print("Selected winner: v2_role_constraints")
print(winning_prompt)

,Prompt,Correct,Accuracy
0,v1_zero_shot,10/10,100%
1,v2_role_constraints,10/10,100%
2,v3_few_shot,10/10,100%
3,v4_step_by_step,9/10,90%


Selected winner: v2_role_constraints

You are an agricultural support classification assistant.
Classify the complaint into exactly one category:
PEST, IRRIGATION, SEED_QUALITY, OTHER.

Rules:
- PEST: insects, pests, crop diseases.
- IRRIGATION: water supply, pumps, pipes.
- SEED_QUALITY: damaged seeds, poor germination.
- OTHER: anything unrelated to the above.
Never invent a category. Return only the label.

Complaint: {text}

